In [2]:

import pandas as pd             # data package
import matplotlib.pyplot as plt # graphics 
import datetime as dt
import numpy as np

import requests, io             # internet and input tools  
import zipfile as zf            # zip file tools 
import os  

#import weightedcalcs as wc
#import numpy as np

import pyarrow as pa
import pyarrow.parquet as pq

from concordance import build_concordance, aggregate_to_bea
from compute_tariff_rates import load_naics_imports, compute_effective_tariff_rates



In [3]:
df = load_naics_imports('./data/imports/TOTALnaics-data-2025-12.parquet')

rates = compute_effective_tariff_rates(df, '2025-07')

In [4]:
# ── Step 3: build concordance from NAICS6 codes in your data ─────────────────
concordance = build_concordance(rates['naics6'].tolist())

# ── Step 4: aggregate to BEA IO level ────────────────────────────────────────
bea_rates = aggregate_to_bea(rates, concordance)

# ── Step 5: inspect output ────────────────────────────────────────────────────
print("\nFull BEA IO tariff rates:")
print(bea_rates[['bea_io', 'bea_desc', 'tau', 'tau_imputed']].to_string(index=False))

print(f"\nShape: {bea_rates.shape}")
print(f"Imputed zeros: {bea_rates['tau_imputed'].sum()}")
print(f"Tau range: {bea_rates['tau'].min():.4f} – {bea_rates['tau'].max():.4f}")

Concordance built: 384/388 NAICS6 codes mapped to BEA IO industries
  Out of scope (services/unmapped): 4 codes
  BEA IO industries covered: 22

Import value coverage: 94.9% of total import value mapped to BEA IO industries

BEA IO industries with no tariff data (tau set to 0):
bea_io                      bea_desc
   213 Support activities for mining

Full BEA IO tariff rates:
bea_io                                         bea_desc      tau  tau_imputed
 111CA                                            Farms 0.060092        False
 113FF        Forestry, fishing, and related activities 0.073363        False
   211                           Oil and gas extraction 0.000152        False
   212                       Mining, except oil and gas 0.020474        False
   213                    Support activities for mining 0.000000         True
 311FT           Food and beverage and tobacco products 0.086397        False
 313TT          Textile mills and textile product mills 0.227196        Fa

In [6]:
bea_rates.head(20)

,bea_io,bea_desc,time,imports,duties,tau,tau_imputed
0,111CA,Farms,2025-07-01,5.022556e+09,3.018139e+08,0.060092,False
1,113FF,"Forestry, fishing, and related activities",2025-07-01,1.686384e+09,1.237179e+08,0.073363,False
2,211,Oil and gas extraction,2025-07-01,1.274354e+10,1.940821e+06,0.000152,False
3,212,"Mining, except oil and gas",2025-07-01,6.006551e+08,1.229800e+07,0.020474,False
4,213,Support activities for mining,2025-07-01,0.000000e+00,0.000000e+00,0.000000,True
5,311FT,Food and beverage and tobacco products,2025-07-01,1.345943e+10,1.162861e+09,0.086397,False
6,313TT,Textile mills and textile product mills,2025-07-01,2.649164e+09,6.018807e+08,0.227196,False
7,315AL,Apparel and leather and allied products,2025-07-01,1.148971e+10,2.970287e+09,0.258517,False
8,321,Wood products,2025-07-01,2.272228e+09,1.205232e+08,0.053042,False
9,322,Paper products,2025-07-01,2.172579e+09,2.047549e+08,0.094245,False
